In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/NYC_Taxi_Cleaned_Analysis_Ready.csv')

In [4]:
# 1. Drop rows with anomalous pickup years
df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
df = df[df['tpep_pickup_datetime'].dt.year == 2025].copy()
print(f"Rows after dropping anomalous years: {len(df)}")

# 2. Clean missing fees
df['Airport_fee'] = df['Airport_fee'].fillna(0)

# 3. Revenue & Fee Impact Metrics (For Q1, Q2, Q3)
# Calculate total fees and the true net revenue
df['total_fees'] = df['cbd_congestion_fee'] + df['congestion_surcharge'] + df['Airport_fee']
df['net_revenue'] = df['total_amount'] - df['total_fees']

# Calculate Net Revenue Per Occupied Driving Hour
df['net_revenue_per_hour'] = (df['net_revenue'] / df['duration_min']) * 60

# Calculate the % of gross revenue eaten by fees (Extent of reduction for Q2)
df['fee_impact_pct'] = (df['total_fees'] / df['total_amount']) * 100

# 4. Engineer Fare Tiers & Trip Length Thresholds (For Q2)
# Using qcut to divide fares and distances into 3 equal-sized distribution buckets
df['fare_tier'] = pd.qcut(df['total_amount'], q=3, labels=['Low Fare', 'Medium Fare', 'High Fare'])
df['trip_length_tier'] = pd.qcut(df['trip_distance'], q=3, labels=['Short Trip', 'Medium Trip', 'Long Trip'])

# 5. Engineer Traffic Congestion Levels (For Q3)
# Defining traffic by speed limits/averages (0-8mph = Heavy, 8-15mph = Moderate, 15mph+ = Light)
speed_bins = [0, 8, 15, 100]
speed_labels = ['Heavy Traffic', 'Moderate Traffic', 'Light Traffic']
df['congestion_level'] = pd.cut(df['speed_mph'], bins=speed_bins, labels=speed_labels)

# 6. Preview the variables needed for final analysis
cols_to_show = [
    'day_of_week', 'pickup_hour', 'PULocationID',
    'fare_tier', 'trip_length_tier', 'congestion_level',
    'fee_impact_pct', 'net_revenue_per_hour'
]

print("--- ABT ---")
display(df[cols_to_show].head(10))

Rows after dropping anomalous years: 3946695
--- ABT ---


,day_of_week,pickup_hour,PULocationID,fare_tier,trip_length_tier,congestion_level,fee_impact_pct,net_revenue_per_hour
0,Saturday,20,238,Low Fare,Short Trip,Heavy Traffic,0.000000,109.729977
1,Saturday,11,50,Low Fare,Short Trip,Heavy Traffic,30.805687,85.882353
2,Tuesday,15,236,Low Fare,Short Trip,Heavy Traffic,15.015015,89.211909
3,Saturday,22,231,Medium Fare,Medium Trip,Moderate Traffic,15.954836,119.210832
4,Saturday,23,137,High Fare,Long Trip,Light Traffic,0.000000,238.101695
5,Wednesday,0,140,Low Fare,Short Trip,Moderate Traffic,16.149871,109.948235
6,Thursday,19,170,Low Fare,Short Trip,Heavy Traffic,18.105850,82.558502
7,Thursday,15,43,Low Fare,Short Trip,Heavy Traffic,20.123839,69.417040
8,Thursday,11,100,Medium Fare,Medium Trip,Heavy Traffic,13.800425,76.603774
9,Sunday,15,163,Low Fare,Short Trip,Heavy Traffic,20.123839,70.792683


In [5]:
df.to_csv('../data/NYC_Taxi_ABT.csv', index=False)